# Week 5 Hands-On Lab — Grounded or Hallucinated? Probing a VLM

**ESP3201 · formative hands-on lab.** Best on free-tier Colab with a **GPU (T4) runtime** for the local model, or use the **hosted-API** path. An **offline mock** path lets you validate the pipeline with no model at all.

You synthesize scenes with **exact ground truth** and probe a VLM three ways:
1. **Easy probe** — present vs random-absent objects (hallucination + recall).
2. **Hard probe** — *binding traps*: ask for a `red circle` when the scene has a red square and a blue circle (color and shape both present, but not together). VLMs hallucinate these far more.
3. **Prompt sensitivity** — the same question in different wordings; a grounded model answers all the same.

> **Report only what your run actually produced.** A capable model can be *well-behaved* on these clean shapes — if so, that is a finding about your evaluation inputs, not a pass.

**Attribution.** Present/absent probing follows **POPE** (Li et al., 2023). Binding-error and prompt-sensitivity framing are standard VLM-evaluation practice. Code is original to ESP3201.

## Setup

In [ ]:
import os
os.system('pip install -q pillow')

# --- Week 5 lab core, embedded directly (no repo clone) -----------------------
# Canonical source, kept in sync by hand: starter/vlm_lab.py in the course
# repo. Cloning a support module from Colab is fragile -- if a session already
# ran once before an update landed, the clone silently no-ops onto the stale
# cached copy instead of fetching the fix. Pasting the needed code directly
# here removes that whole failure class.

import hashlib
from dataclasses import dataclass
from typing import Dict, List, Optional, Set, Tuple

COLORS = {
    "red": (220, 40, 40),
    "blue": (40, 80, 220),
    "green": (40, 170, 70),
    "yellow": (235, 205, 40),
}
SHAPES = ["square", "circle", "triangle"]
VOCAB: List[str] = [f"{c} {s}" for c in COLORS for s in SHAPES]


# --------------------------------------------------------------------------- #
# Scene synthesis (exact ground truth)
# --------------------------------------------------------------------------- #
def synthesize_scene(seed: int, n_objects: int = 3, size: int = 256):
    """Return (PIL.Image, present_objects:set[str]) with exact ground truth."""
    from PIL import Image, ImageDraw  # local import so import-time stays light
    import random

    rng = random.Random(seed)
    chosen = rng.sample(VOCAB, n_objects)
    img = Image.new("RGB", (size, size), (245, 245, 245))
    draw = ImageDraw.Draw(img)
    slots = [(size * (i + 1) // (n_objects + 1), size // 2) for i in range(n_objects)]
    for (label, (cx, cy)) in zip(chosen, slots):
        color_name, shape = label.split(" ", 1)
        col = COLORS[color_name]
        r = size // 8
        if shape == "square":
            draw.rectangle([cx - r, cy - r, cx + r, cy + r], fill=col)
        elif shape == "circle":
            draw.ellipse([cx - r, cy - r, cx + r, cy + r], fill=col)
        else:  # triangle
            draw.polygon([(cx, cy - r), (cx - r, cy + r), (cx + r, cy + r)], fill=col)
    return img, set(chosen)


def build_probe(present: Set[str], n_absent: int = 3, seed: int = 0
                ) -> List[Tuple[str, bool]]:
    """Return [(object, ground_truth_present)] mixing present and absent objects."""
    import random
    rng = random.Random(seed)
    absent_pool = [o for o in VOCAB if o not in present]
    absent = rng.sample(absent_pool, min(n_absent, len(absent_pool)))
    probe = [(o, True) for o in present] + [(o, False) for o in absent]
    rng.shuffle(probe)
    return probe


def build_binding_probe(present: Set[str], seed: int = 0
                        ) -> List[Tuple[str, bool]]:
    """A harder probe targeting binding errors. The absent objects are combos
    whose COLOR and SHAPE both appear in the scene but not together
    (e.g. scene has a 'red square' and a 'blue circle' -> ask about a
    'red circle'). VLMs hallucinate these far more than fully-absent objects,
    so the failure mode shows up even on a capable model."""
    import random
    rng = random.Random(seed)
    colors = {o.split(" ", 1)[0] for o in present}
    shapes = {o.split(" ", 1)[1] for o in present}
    traps = [f"{c} {s}" for c in colors for s in shapes if f"{c} {s}" not in present]
    rng.shuffle(traps)
    probe = [(o, True) for o in present] + [(o, False) for o in traps]
    rng.shuffle(probe)
    return probe


# Different phrasings of the SAME yes/no question. A grounded model answers all
# of them the same way; prompt sensitivity is when the answer flips with wording.
QUESTION_TEMPLATES = [
    "Is there a {obj} in the image? Answer yes or no.",
    "Can you see a {obj}? Answer yes or no.",
    "Does this image contain a {obj}? Answer yes or no.",
    "Is a {obj} present in this picture? Answer yes or no.",
]


# --------------------------------------------------------------------------- #
# Backends
# --------------------------------------------------------------------------- #
class MockVLM:
    """Offline backend. Answers correctly on present objects; on absent objects
    answers 'yes' with probability `yes_bias` (deterministically per question).
    Use this to validate the scoring pipeline before loading a real model."""

    def __init__(self, yes_bias: float = 0.4):
        self.yes_bias = yes_bias

    def answer(self, image, question: str, obj: str, present: bool) -> str:
        if present:
            return "yes"
        h = int(hashlib.sha256(f"{obj}|{question}".encode()).hexdigest(), 16)
        return "yes" if (h % 1000) / 1000.0 < self.yes_bias else "no"


class HFVLM:
    """Small local vision-language model via the standard transformers
    image-text-to-text interface (chat template + generate).

    Verified on `HuggingFaceTB/SmolVLM-Instruct` (see pinned_models.md). The
    same code path works for other instruct VLMs exposed through
    AutoModelForImageTextToText (SmolVLM2, Idefics, Qwen2-VL, ...). Pin and
    re-measure VRAM if you switch checkpoints.
    """

    def __init__(self, model_id: str = "HuggingFaceTB/SmolVLM-Instruct",
                 device: str = "cuda", dtype="float16"):
        import torch
        from transformers import AutoProcessor, AutoModelForImageTextToText
        self.processor = AutoProcessor.from_pretrained(model_id)
        self.model = AutoModelForImageTextToText.from_pretrained(
            model_id, dtype=getattr(torch, dtype)).to(device)
        self.device = device

    def answer(self, image, question: str, obj: str = "", present: bool = False) -> str:
        messages = [{"role": "user", "content": [
            {"type": "image"}, {"type": "text", "text": question}]}]
        prompt = self.processor.apply_chat_template(messages, add_generation_prompt=True)
        inputs = self.processor(text=prompt, images=[image], return_tensors="pt").to(self.device)
        n_in = inputs["input_ids"].shape[1]
        out = self.model.generate(**inputs, max_new_tokens=8, do_sample=False)
        text = self.processor.batch_decode(
            out[:, n_in:], skip_special_tokens=True)[0]
        return _parse_yes_no(text)


class GeminiVLM:
    """Free-tier hosted VLM via google-generativeai.

    PIN THIS: confirm the exact model id (e.g. a current Gemini Flash) and that
    your free-tier key works. Reads GOOGLE_API_KEY from the environment / Colab
    secret.
    """

    def __init__(self, model_id: str = "", api_key: Optional[str] = None):  # PIN THIS
        import os
        import google.generativeai as genai
        if not model_id:
            raise ValueError("Set model_id to a pinned Gemini model (see notebook).")
        genai.configure(api_key=api_key or os.environ["GOOGLE_API_KEY"])
        self.model = genai.GenerativeModel(model_id)

    def answer(self, image, question: str, obj: str = "", present: bool = False) -> str:
        resp = self.model.generate_content([question, image])
        return _parse_yes_no(resp.text)


def _parse_yes_no(text: str) -> str:
    t = (text or "").strip().lower()
    if t.startswith("yes") or " yes" in t[:12]:
        return "yes"
    if t.startswith("no") or " no" in t[:12]:
        return "no"
    return "yes" if "yes" in t else "no"


# --------------------------------------------------------------------------- #
# Run + score
# --------------------------------------------------------------------------- #
@dataclass
class ProbeRecord:
    scene_seed: int
    obj: str
    present: bool
    answered_yes: bool


def run_probe(backend, n_scenes: int = 8, n_objects: int = 3,
              n_absent: int = 3, base_seed: int = 0,
              hard: bool = False) -> List[ProbeRecord]:
    """Run the present/absent probe over synthesized scenes.

    hard=False: absent objects are random combos not in the scene.
    hard=True:  absent objects are binding traps (color and shape both present
                but not together) -- elicits hallucination even on good models.
    """
    records: List[ProbeRecord] = []
    for i in range(n_scenes):
        seed = base_seed + i
        image, present = synthesize_scene(seed, n_objects=n_objects)
        probe = (build_binding_probe(present, seed=seed) if hard
                 else build_probe(present, n_absent=n_absent, seed=seed))
        for obj, gt in probe:
            q = f"Is there a {obj} in the image? Answer yes or no."
            ans = backend.answer(image, q, obj=obj, present=gt)
            records.append(ProbeRecord(seed, obj, gt, ans == "yes"))
    return records


def score(records: List[ProbeRecord]) -> Dict[str, float]:
    present = [r for r in records if r.present]
    absent = [r for r in records if not r.present]
    tp = sum(r.answered_yes for r in present)
    fp = sum(r.answered_yes for r in absent)
    n_present = len(present)
    n_absent = len(absent)
    yes_total = sum(r.answered_yes for r in records)
    correct = tp + (n_absent - fp)
    return {
        "n_questions": len(records),
        "accuracy": round(correct / max(1, len(records)), 3),
        "hallucination_rate": round(fp / max(1, n_absent), 3),   # yes on ABSENT
        "recall_present": round(tp / max(1, n_present), 3),      # yes on PRESENT
        "yes_rate": round(yes_total / max(1, len(records)), 3),  # overall yes-bias
        "precision": round(tp / max(1, tp + fp), 3),
    }


# --------------------------------------------------------------------------- #
# Prompt-sensitivity probe (a signature VLM limitation, independent of whether
# the model hallucinates objects)
# --------------------------------------------------------------------------- #
@dataclass
class SensitivityRecord:
    scene_seed: int
    obj: str
    present: bool
    answers: List[bool]          # one per question template
    flipped: bool                # did the answer change with phrasing?


def run_prompt_sensitivity(backend, n_scenes: int = 4, n_objects: int = 3,
                           n_absent: int = 3, base_seed: int = 0
                           ) -> List[SensitivityRecord]:
    """Ask the SAME object question under several phrasings and record whether
    the answer flips. A well-grounded model is phrasing-invariant."""
    records: List[SensitivityRecord] = []
    for i in range(n_scenes):
        seed = base_seed + i
        image, present = synthesize_scene(seed, n_objects=n_objects)
        for obj, gt in build_probe(present, n_absent=n_absent, seed=seed):
            answers = []
            for tmpl in QUESTION_TEMPLATES:
                ans = backend.answer(image, tmpl.format(obj=obj), obj=obj, present=gt)
                answers.append(ans == "yes")
            records.append(SensitivityRecord(seed, obj, gt, answers,
                                             flipped=len(set(answers)) > 1))
    return records


def score_prompt_sensitivity(records: List[SensitivityRecord]) -> Dict[str, float]:
    present = [r for r in records if r.present]
    absent = [r for r in records if not r.present]

    def flip_rate(rs):
        return round(sum(r.flipped for r in rs) / max(1, len(rs)), 3)

    return {
        "n_objects": len(records),
        "n_templates": len(QUESTION_TEMPLATES),
        "flip_rate_overall": flip_rate(records),     # answer changed with wording
        "flip_rate_present": flip_rate(present),
        "flip_rate_absent": flip_rate(absent),
    }


# --------------------------------------------------------------------------- #
# Real-image probe (optional, for a stronger hallucination signal than the
# synthetic shapes give)
# --------------------------------------------------------------------------- #
def load_labeled_images(image_dir: str):
    """Load (PIL image, present_objects:set) pairs from a directory.

    Expects images plus a `labels.json` mapping filename -> list of present
    object strings (using the same `{color} {shape}` vocabulary, or your own).
    Lets an instructor swap in natural photos where VLM hallucination is more
    pronounced than on the synthetic shapes; the metric code is unchanged.
    """
    import json
    import os
    from PIL import Image
    labels = json.load(open(os.path.join(image_dir, "labels.json")))
    items = []
    for fname, present in labels.items():
        items.append((Image.open(os.path.join(image_dir, fname)).convert("RGB"),
                      set(present)))
    return items


def run_probe_on_images(backend, items, vocab=None, n_absent: int = 3,
                        seed: int = 0) -> List[ProbeRecord]:
    """Run the present/absent probe on caller-supplied (image, present) pairs."""
    import random
    vocab = vocab or VOCAB
    rng = random.Random(seed)
    records: List[ProbeRecord] = []
    for idx, (image, present) in enumerate(items):
        absent_pool = [o for o in vocab if o not in present]
        absent = rng.sample(absent_pool, min(n_absent, len(absent_pool)))
        probe = [(o, True) for o in present] + [(o, False) for o in absent]
        for obj, gt in probe:
            q = f"Is there a {obj} in the image? Answer yes or no."
            ans = backend.answer(image, q, obj=obj, present=gt)
            records.append(ProbeRecord(idx, obj, gt, ans == "yes"))
    return records

print('vlm_lab loaded (embedded in this notebook -- no repo clone needed). vocabulary size:', len(VOCAB))

## Look at a synthesized scene, an easy probe, and a binding trap

In [ ]:
img, present = synthesize_scene(seed=3, n_objects=3)
print('present (ground truth):', sorted(present))
print('easy probe :', build_probe(present, n_absent=3, seed=3))
print('binding trap probe:', build_binding_probe(present, seed=3))
img  # displays inline

## Choose a backend

Pick ONE and run its cell.

- **A. Offline mock** — no model; simulates a yes-bias. Good for a dry run.
- **B. Local VLM (T4)** — defaults to `HuggingFaceTB/SmolVLM-Instruct`, **verified to fit a free T4** (~5 GB; see `pinned_models.md`). Swap `MODEL_ID` and re-check memory.
- **C. Hosted API (free tier)** — a Gemini Flash model. **PIN THIS** and supply a key.

In [ ]:
# --- A. Offline mock ---
backend = MockVLM(yes_bias=0.4)   # try 0.0, 0.4, 0.9
print('using MockVLM')

In [ ]:
# --- B. Local VLM on T4 (uncomment to use) ---
# os.system('pip install -q transformers accelerate torch')
# MODEL_ID = 'HuggingFaceTB/SmolVLM-Instruct'   # VERIFIED on T4: ~5 GB fp16
# backend = HFVLM(model_id=MODEL_ID)
# print('using HFVLM', MODEL_ID)

In [ ]:
# --- C. Hosted Gemini (free tier) (uncomment and PIN THIS) ---
# os.system('pip install -q google-generativeai')
# from google.colab import userdata
# os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY')
# GEMINI_MODEL = ''  # PIN THIS, a current free-tier Gemini Flash id
# backend = GeminiVLM(model_id=GEMINI_MODEL)
# print('using GeminiVLM', GEMINI_MODEL)

## Probe 1 + 2 — easy vs binding-trap

Compare hallucination on random-absent objects vs binding traps. If the model is honest on easy absents but hallucinates traps, you have found a binding failure.

In [ ]:
easy = score(run_probe(backend, n_scenes=6, hard=False))
hard = score(run_probe(backend, n_scenes=6, hard=True))
print('EASY (random absent):', easy)
print('HARD (binding traps):', hard)

### See it, don't just read it

`easy`/`hard` above are printed dicts -- easy to skim past. Below: the same probe, but with the
model's actual answer shown next to each object on the scene it was asked about, and the two
conditions plotted side by side.

In [ ]:
import matplotlib.pyplot as plt

def probe_with_answers(backend, seed, n_objects=3, hard=False):
    img, present = synthesize_scene(seed, n_objects=n_objects)
    probe = build_binding_probe(present, seed=seed) if hard else build_probe(present, n_absent=3, seed=seed)
    rows = []
    for obj, gt in probe:
        q = f"Is there a {obj} in the image? Answer yes or no."
        answered_yes = backend.answer(img, q, obj=obj, present=gt) == "yes"
        if gt and answered_yes:
            status, color = "correct (hit)", "#2a7a2a"
        elif gt and not answered_yes:
            status, color = "MISSED", "#b8860b"
        elif not gt and answered_yes:
            status, color = "HALLUCINATED", "#c0392b"
        else:
            status, color = "correct (rejected)", "#2a7a2a"
        rows.append((obj, gt, answered_yes, status, color))
    return img, present, rows

def show_scene_probe(backend, seed, n_objects=3, hard=False, title=None):
    img, present, rows = probe_with_answers(backend, seed, n_objects=n_objects, hard=hard)
    fig, axes = plt.subplots(1, 2, figsize=(8, 3.6), gridspec_kw={"width_ratios": [1, 1.3]})
    axes[0].imshow(img)
    axes[0].axis("off")
    axes[0].set_title(f"scene seed={seed}", fontsize=10)
    axes[1].axis("off")
    axes[1].text(0, 1.0, f"ground truth present: {', '.join(sorted(present))}", fontsize=9, va="top")
    y = 0.86
    for obj, gt, answered_yes, status, color in rows:
        label = (f"{'present' if gt else 'absent ':8s} {obj:16s} -> model said "
                 f"{'yes' if answered_yes else 'no ':3s} [{status}]")
        axes[1].text(0, y, label, fontsize=9, color=color, va="top", family="monospace")
        y -= 0.11
    fig.suptitle(title or ("binding-trap probe" if hard else "easy probe"))
    plt.tight_layout()
    plt.show()

show_scene_probe(backend, seed=3, hard=True, title="Binding-trap probe -- one scene, every answer shown")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
metrics = ["hallucination_rate", "recall_present", "accuracy"]
x = range(len(metrics))
w = 0.35
ax.bar([i - w / 2 for i in x], [easy[m] for m in metrics], width=w, label="easy (random absent)", color="#4c72b0")
ax.bar([i + w / 2 for i in x], [hard[m] for m in metrics], width=w, label="hard (binding traps)", color="#c44e52")
ax.set_xticks(list(x))
ax.set_xticklabels(metrics, rotation=15, ha="right")
ax.set_ylim(0, 1)
ax.set_title("Easy probe vs. binding-trap probe")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

## Probe 3 — prompt sensitivity

Ask the SAME object question four ways. `flip_rate` > 0 means the answer depends on wording, not just on the image.

In [ ]:
sens = run_prompt_sensitivity(backend, n_scenes=3, n_objects=3, n_absent=3)
print('PROMPT SENSITIVITY:', score_prompt_sensitivity(sens))
for r in [x for x in sens if x.flipped][:5]:
    print(f'  FLIP {r.obj} (present={r.present}) answers={r.answers}')

### Watch a question flip with nothing but the wording

Same image, same object, four ways of asking. A grounded model answers all four the same way --
below is one case where it didn't.

In [ ]:
def show_prompt_sensitivity(backend, seed, obj=None, n_objects=3):
    img, present = synthesize_scene(seed, n_objects=n_objects)
    probe = build_probe(present, n_absent=3, seed=seed)
    if obj is None:
        # default to the first object whose answer actually flips, for a demonstrative example
        for candidate, gt in probe:
            answers = [backend.answer(img, t.format(obj=candidate), obj=candidate, present=gt)
                       for t in QUESTION_TEMPLATES]
            if len(set(answers)) > 1:
                obj = candidate
                break
        else:
            obj = probe[0][0]
    gt = dict(probe)[obj]
    answers = [backend.answer(img, t.format(obj=obj), obj=obj, present=gt) for t in QUESTION_TEMPLATES]
    majority = max(set(answers), key=answers.count)

    fig, axes = plt.subplots(1, 2, figsize=(9, 3.8), gridspec_kw={"width_ratios": [1, 1.4]})
    axes[0].imshow(img)
    axes[0].axis("off")
    axes[0].set_title(f"scene seed={seed}", fontsize=10)
    axes[1].axis("off")
    y = 1.0
    for tmpl, ans in zip(QUESTION_TEMPLATES, answers):
        color = "#2a7a2a" if ans == majority else "#c0392b"
        flip_tag = "" if ans == majority else "  <-- FLIPPED"
        axes[1].text(0, y, f'"{tmpl.format(obj=obj)}"', fontsize=8.5, va="top")
        axes[1].text(0, y - 0.09, f"  -> {ans}{flip_tag}", fontsize=9, color=color, va="top",
                     family="monospace", fontweight="bold")
        y -= 0.24
    fig.suptitle(f'Asking about "{obj}" (ground truth present={gt}) -- same image, four phrasings',
                 fontsize=11)
    plt.tight_layout(rect=(0, 0, 1, 0.93))
    plt.show()
    return obj, answers

_ = show_prompt_sensitivity(backend, seed=1)

In [ ]:
sens_score = score_prompt_sensitivity(sens)
fig, ax = plt.subplots(figsize=(5, 3.6))
flip_metrics = ["flip_rate_present", "flip_rate_absent", "flip_rate_overall"]
ax.bar(range(len(flip_metrics)), [sens_score[m] for m in flip_metrics], color="#8172b2")
ax.set_xticks(range(len(flip_metrics)))
ax.set_xticklabels(flip_metrics, rotation=15, ha="right")
ax.set_ylim(0, 1)
ax.set_title("Prompt-sensitivity flip rate\n(answer changes with wording alone)")
plt.tight_layout()
plt.show()

### Checkpoint — is this specific to a mock backend?

`MockVLM`'s hallucination is a deterministic hash of the *question text plus the object*, so a
present object always gets "yes" (checked first, before the hash) while an absent object's answer
depends on exactly how the question is worded -- that is precisely why `flip_rate_present` is 0
and `flip_rate_absent` is high above. A real VLM won't have this exact mechanism, but the
*symptom* -- present objects answered consistently, absent-object answers wobbling with phrasing
-- shows up empirically in real hallucination studies too. If you swap in `HFVLM`, rerun both
plots above and compare: does the flip pattern look similar, or does presence also start flipping?

**What would an actual fix look like?** You can't just "turn off yes_bias" in a real model the way
this demo can — hallucination isn't a single knob. Practical mitigations used in real VLM systems
include: majority-voting the SAME question across several paraphrases and only trusting the
answer if it's stable (this probe *is* that check, run manually); grounding the answer in a
retrieved caption or detector output instead of the model's free-form judgment; calibration /
abstention (let the model say "unsure" instead of forcing yes/no); and chain-of-verification
(ask the model to re-derive its answer from a description of what it sees, then check
consistency). None of these make hallucination zero -- they trade it for cost, latency, or
occasional over-caution.

## Inspect the failures that did occur

In [ ]:
recs = run_probe(backend, n_scenes=6, hard=True)
halluc = [r for r in recs if (not r.present) and r.answered_yes]
missed = [r for r in recs if r.present and not r.answered_yes]
print(f'{len(halluc)} hallucinated (yes on absent):', [(r.scene_seed, r.obj) for r in halluc[:6]])
print(f'{len(missed)} missed (no on present):    ', [(r.scene_seed, r.obj) for r in missed[:6]])

## Optional — real images for a stronger hallucination signal

Clean synthetic shapes are *easy*; a good VLM may not hallucinate on them at all. To push harder, drop a few natural photos into a folder with a `labels.json` (`filename -> [present objects]`) and re-run the same metric:

```python
items = load_labeled_images('my_images')           # PIL images + ground truth
print(score(run_probe_on_images(backend, items)))
```

## Worksheet (your deliverable)

### 1. Grounding table

| Probe | Accuracy | Hallucination rate | Recall (present) | Yes-rate |
|-------|----------|--------------------|------------------|----------|
| easy | | | | |
| binding traps | | | | |

Prompt-sensitivity flip rate (overall / present / absent): ____ / ____ / ____

### 2. What failed, and what didn't

- Did the model **hallucinate** (yes on absent)? Did binding traps change that vs easy absents?
- Did it **miss present objects** (recall < 1)? Did its answer **flip with phrasing**?

### 3. The evaluation-input lesson

- If a failure mode did **not** appear, was that because the model is good, or because the synthetic shapes are too easy? What input *would* expose it? (Try the real-image cell.)

### 4. When to trust it

- Is the model yes-biased (hallucinates) or no-biased (misses real objects)? What does each imply for a robot's perception module?
- `One deployment guardrail you would add before trusting this VLM:`

## AI-Agent Usage Disclosure

State:

- which tools you used
- what they helped produce
- what you verified or rewrote yourself
- one specific thing you did not trust without checking